# Estimation Analysis - Sleep Health Dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load dataset
url = 'https://raw.githubusercontent.com/Muhanad-husn/Sleep-Health-and-Lifestyle/refs/heads/main/data.csv'
df = pd.read_csv(url)
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## 1. Sample statistics of numerical variables

In [2]:
df.describe()

,Person ID,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps
count,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000
mean,187.500000,42.184492,7.132086,7.312834,59.171123,5.385027,70.165775,6816.844920
std,108.108742,8.673133,0.795657,1.196956,20.830804,1.774526,4.135676,1617.915679
min,1.000000,27.000000,5.800000,4.000000,30.000000,3.000000,65.000000,3000.000000
25%,94.250000,35.250000,6.400000,6.000000,45.000000,4.000000,68.000000,5600.000000
50%,187.500000,43.000000,7.200000,7.000000,60.000000,5.000000,70.000000,7000.000000
75%,280.750000,50.000000,7.800000,8.000000,75.000000,7.000000,72.000000,8000.000000
max,374.000000,59.000000,8.500000,9.000000,90.000000,8.000000,86.000000,10000.000000


## 2. Population parameters (assuming dataset is population)

In [3]:
population_mean = df.mean(numeric_only=True)
population_std = df.std(numeric_only=True)

population_mean, population_std

(Person ID                   187.500000
 Age                          42.184492
 Sleep Duration                7.132086
 Quality of Sleep              7.312834
 Physical Activity Level      59.171123
 Stress Level                  5.385027
 Heart Rate                   70.165775
 Daily Steps                6816.844920
 dtype: float64,
 Person ID                   108.108742
 Age                           8.673133
 Sleep Duration                0.795657
 Quality of Sleep              1.196956
 Physical Activity Level      20.830804
 Stress Level                  1.774526
 Heart Rate                    4.135676
 Daily Steps                1617.915679
 dtype: float64)

## 3. Simple random sample (n=30)

In [4]:
srs = df.sample(n=30, random_state=42)
srs['Sleep Duration'].mean()

np.float64(6.930000000000001)

## 4. Systematic sample (interval=5)

In [5]:
systematic = df.iloc[::5].head(30)
systematic['Sleep Duration'].mean()

np.float64(6.823333333333333)

## 5. Stratified sample (Occupation)

In [14]:
stratified = df.groupby('Occupation').apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)
stratified['Sleep Duration'].mean()

C:\Users\kunal\AppData\Local\Temp\ipykernel_6928\2510092908.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stratified = df.groupby('Occupation').apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)


np.float64(6.945454545454546)

## 6. Estimate population mean

In [7]:
df['Sleep Duration'].mean()

np.float64(7.132085561497325)

## 7. Estimate population proportion (BMI Category)

In [8]:
sample_30 = df.sample(30, random_state=42)
sample_30['BMI Category'].value_counts(normalize=True)

BMI Category
Normal        0.633333
Overweight    0.366667
Name: proportion, dtype: float64

## 8. Minimum sample size estimation

In [9]:
# Using formula n = (Z*sigma/E)^2
Z = 1.96
sigma = df['Sleep Duration'].std()
E = 0.2

n = (Z * sigma / E) ** 2
int(np.ceil(n))

61

## 9. 95% CI (known sigma)

In [10]:
sample_40 = df.sample(40, random_state=42)
mean = sample_40['Sleep Duration'].mean()
sigma = 0.8
n = 40

margin = 1.96 * (sigma / np.sqrt(n))
(mean - margin, mean + margin)

(np.float64(6.787077431442798), np.float64(7.2829225685572005))

## 10. 95% CI (unknown sigma, n=40)

In [11]:
std = sample_40['Sleep Duration'].std()
margin = stats.t.ppf(0.975, df=39) * (std / np.sqrt(40))
(mean - margin, mean + margin)

(np.float64(6.767467252032871), np.float64(7.302532747967128))

## 11. 95% CI (n=20)

In [12]:
sample_20 = df.sample(20, random_state=42)
mean20 = sample_20['Sleep Duration'].mean()
std20 = sample_20['Sleep Duration'].std()

margin = stats.t.ppf(0.975, df=19) * (std20 / np.sqrt(20))
(mean20 - margin, mean20 + margin)

(np.float64(6.500858550382792), np.float64(7.379141449617207))

## 12. 95% CI for proportion (BMI Normal)

In [13]:
p = (sample_30['BMI Category'] == 'Normal').mean()
n = 30

margin = 1.96 * np.sqrt((p * (1 - p)) / n)
(p - margin, p + margin)

(np.float64(0.46088963344948825), np.float64(0.8057770332171783))